# AMD AI Challenge — Mini-Challenge 2: OCR (licence plates & road signs)

This notebook builds a complete, submission-ready solution:

| Step | What happens |
|---|---|
| 1 | Config + GPU check |
| 2 | Writes the submission project (`app.py`, `ocr_core.py`, `server.py`, `Dockerfile`, `requirements.txt`, …) |
| 3 | Installs dependencies and downloads the VLM weights to **persistent storage** |
| 4 | Recreates the 10 sample images from the challenge PDF (PNG / JPEG / TIFF) |
| 5 | Scores the model on the samples using the **official normalization** |
| 6 | Robustness test (blur, noise, glare, low light, skew, huge/tiny images) |
| 7 | End-to-end test of the exact harness call: `python3 /app/app.py --input-image …` with timing + VRAM |
| 8 | Packages the project as a zip → build Docker image → push → submit (guide at the bottom) |

**Approach:** a vision-language model (default **Qwen3-VL-8B-Instruct**, bf16, ~17 GB VRAM) with a strict
transcription prompt + deterministic post-processing, plus an image-enhancement retry for dark/noisy images.

**Key design point — the harness runs `app.py` as a new process for every image (30 s limit each).**
So the container's `CMD` starts `server.py`, which loads the model **once** during the 10-minute startup window
and keeps it on the GPU. `app.py` is a thin client that asks the server and writes the JSON
(~1–3 s per image). If the server isn't running for some reason, `app.py` launches it and waits.

**Weights are not in the image** (small, fast push). The server downloads them from Hugging Face at container
start (allowed: the container has network access during evaluation). `app.py` never exceeds the 30 s limit:
if the model is still not ready it writes an empty answer for that image instead of timing out the whole run.

> Run cells top to bottom. Remember: 3 h/day GPU quota — press **Turn-off Session** when done.

## 1. Config

In [ ]:
import os, sys, pathlib, subprocess, shutil, json, time

# Persistent storage depends on the pod URL: "jupyter-hack-***" -> /persistent, "rgapi-hackathon-***" -> /workspace
def pick_persistent():
    for d in ("/persistent", "/workspace"):
        if os.path.isdir(d) and os.access(d, os.W_OK):
            return pathlib.Path(d)
    return pathlib.Path.cwd()

PERSIST   = pick_persistent()
PROJ      = PERSIST / "ocr_submission"          # becomes /app in the Docker image
MODEL_ID  = "Qwen/Qwen3-VL-8B-Instruct"          # alternatives: "Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen3-VL-4B-Instruct"
MODEL_DIR = PERSIST / "models" / MODEL_ID.split("/")[-1]
SAMPLES   = PROJ / "samples"
OUTDIR    = PROJ / "output"

for d in (PROJ, SAMPLES, OUTDIR, MODEL_DIR.parent):
    d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]         = str(PERSIST / "hf_home")   # keep HF cache off the ephemeral disk
os.environ["OCR_MODEL_ID"]    = MODEL_ID
os.environ["OCR_MODEL_DIR"]   = str(MODEL_DIR)
os.environ["OCR_OUTPUT_DIR"]  = str(OUTDIR)
os.chdir(PROJ)
print("persistent:", PERSIST, "\nproject   :", PROJ, "\nmodel dir :", MODEL_DIR)

In [ ]:
import torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| HIP", getattr(torch.version, "hip", None))
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(p.name, f"{p.total_memory/2**30:.1f} GiB")
!(amd-smi static --asic 2>/dev/null | head -n 15) || rocm-smi --showproductname 2>/dev/null
!df -h {PERSIST} | tail -n 1

## 2. Write the submission project

These files become `/app` inside the Docker image. Edit them here and re-run the cell to change behaviour.

In [ ]:
%%writefile requirements.txt
# torch / torchvision / PIL come from the mandated ROCm base image - do NOT list torch here
# (pip would replace the ROCm build with a CUDA build).
transformers>=4.57.0
accelerate>=1.0.0
huggingface_hub>=0.34.0
hf_xet          # fast parallel downloads from the Hugging Face Hub
safetensors
numpy

In [ ]:
%%writefile ocr_core.py
"""Shared OCR logic: image loading, model, prompt, post-processing."""
import os, re, math, unicodedata
import numpy as np
from PIL import Image, ImageOps, ImageFilter, ImageStat

Image.MAX_IMAGE_PIXELS = None  # test images can be very large
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")   # max-speed weight download at container start

MODEL_ID  = os.environ.get("OCR_MODEL_ID", "Qwen/Qwen3-VL-8B-Instruct")
MODEL_DIR = os.environ.get("OCR_MODEL_DIR", "/models/ocr-vlm")
MAX_SIDE  = int(os.environ.get("OCR_MAX_SIDE", "1280"))
MIN_SIDE  = int(os.environ.get("OCR_MIN_SIDE", "448"))
MAX_NEW_TOKENS = int(os.environ.get("OCR_MAX_NEW_TOKENS", "48"))

PROMPT = """You are a precise OCR engine for traffic-camera images. Transcribe the text in this image.
Reply with ONLY the transcription: no explanation, no labels, no quotes, no punctuation you did not see.

Rules:
1. Licence plate: output ONLY the registration number. Drop everything printed around it: state or
   province names (CALIFORNIA, NEW YORK, TEXAS ...), slogans and mottos (EXCELSIOR, THE LONE STAR STATE ...),
   websites, dealer frames and registration stickers.
2. Chinese plate: the leading Chinese province character and the letter after it ARE part of the number.
   Keep them, e.g. 京A·12345 or 沪B·88888.
3. Road sign (stop, speed limit, work zone, warning, advisory plaque): output every word and number printed
   on the sign, e.g. STOP, SPEED LIMIT 65, ROAD WORK AHEAD. If the sign shows only a number, output only
   that number - never add units such as MPH that are not printed.
4. Text on several lines: read top to bottom and join the lines with a single space.
5. Copy every character exactly as printed. The image may be blurry, noisy, dark, glared or seen at an angle -
   look carefully at look-alike characters (0/O/D/Q, 1/I/L, 2/Z, 5/S, 6/G, 8/B).
"""

US_BANNERS = [
    "ALABAMA","ALASKA","ARIZONA","ARKANSAS","CALIFORNIA","COLORADO","CONNECTICUT","DELAWARE","FLORIDA",
    "GEORGIA","HAWAII","IDAHO","ILLINOIS","INDIANA","IOWA","KANSAS","KENTUCKY","LOUISIANA","MAINE",
    "MARYLAND","MASSACHUSETTS","MICHIGAN","MINNESOTA","MISSISSIPPI","MISSOURI","MONTANA","NEBRASKA",
    "NEVADA","NEW HAMPSHIRE","NEW JERSEY","NEW MEXICO","NEW YORK","NORTH CAROLINA","NORTH DAKOTA","OHIO",
    "OKLAHOMA","OREGON","PENNSYLVANIA","RHODE ISLAND","SOUTH CAROLINA","SOUTH DAKOTA","TENNESSEE","TEXAS",
    "UTAH","VERMONT","VIRGINIA","WASHINGTON","WEST VIRGINIA","WISCONSIN","WYOMING","DISTRICT OF COLUMBIA",
    "THE LONE STAR STATE","LONE STAR STATE","EXCELSIOR","EMPIRE STATE","THE EMPIRE STATE","SUNSHINE STATE",
    "THE SUNSHINE STATE","GARDEN STATE","LAND OF LINCOLN","GRAND CANYON STATE","THE GOLDEN STATE",
    "GOLDEN STATE","DMV.CA.GOV","FAMOUS POTATOES","LIVE FREE OR DIE","FIRST IN FLIGHT","PEACH STATE",
]
_BANNER_RE = re.compile(r"(?<![A-Z0-9])(" + "|".join(sorted(map(re.escape, US_BANNERS), key=len, reverse=True)) + r")(?![A-Z0-9])")
_PREFIX_RE = re.compile(r"^\s*(the\s+)?(text|answer|transcription|output|result|plate(\s+number)?|licen[cs]e\s+plate(\s+number)?|sign|reading)\s*(is)?\s*[:：\-]\s*", re.I)


# ----------------------------------------------------------------------------- images
def load_image(path):
    im = Image.open(path)
    try:
        im.seek(0)                      # multi-page TIFF -> first page
    except Exception:
        pass
    im.load()
    try:
        im = ImageOps.exif_transpose(im)
    except Exception:
        pass
    if im.mode in ("I;16", "I;16B", "I;16L", "I", "F"):          # 16-bit / float TIFF
        a = np.asarray(im, dtype=np.float32)
        lo, hi = float(a.min()), float(a.max())
        a = (a - lo) / (hi - lo + 1e-6) * 255.0
        im = Image.fromarray(a.clip(0, 255).astype(np.uint8))
    if im.mode in ("RGBA", "LA", "P", "PA"):
        im = im.convert("RGBA")
        bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
        im = Image.alpha_composite(bg, im)
    return fit_size(im.convert("RGB"))


def fit_size(im):
    w, h = im.size
    s = 1.0
    if max(w, h) > MAX_SIDE:
        s = MAX_SIDE / max(w, h)
    elif min(w, h) < MIN_SIDE:
        s = min(MIN_SIDE / min(w, h), MAX_SIDE / max(w, h))
    if abs(s - 1.0) > 1e-3:
        im = im.resize((max(28, round(w * s)), max(28, round(h * s))), Image.LANCZOS)
    return im


def image_stats(im):
    g = im.convert("L")
    st = ImageStat.Stat(g)
    return st.mean[0], st.stddev[0]


def needs_enhance(im):
    mean, std = image_stats(im)
    return mean < 80 or std < 35


def enhance(im):
    """Denoise + stretch contrast + brighten shadows: helps low light, glare and sensor noise."""
    out = im.filter(ImageFilter.MedianFilter(3))
    out = ImageOps.autocontrast(out, cutoff=2)
    mean, _ = image_stats(out)
    if mean < 110:
        gamma = 0.6 if mean < 70 else 0.8
        lut = [round(255 * (i / 255) ** gamma) for i in range(256)]
        out = out.point(lut * 3)
    return out.filter(ImageFilter.UnsharpMask(radius=2, percent=120, threshold=3))


# ----------------------------------------------------------------------------- text
def postprocess(raw):
    t = raw or ""
    t = re.sub(r"<think>.*?</think>", " ", t, flags=re.S)
    t = t.replace("```", " ").strip()
    if t.startswith("{") and '"text"' in t:                       # model answered with JSON
        try:
            import json
            t = str(json.loads(t).get("text", t))
        except Exception:
            pass
    t = unicodedata.normalize("NFKC", t)                           # full-width -> ASCII, keeps CJK
    for ch in "•∙・‧⋅●":
        t = t.replace(ch, "·")
    lines = []
    for ln in t.splitlines():
        ln = _PREFIX_RE.sub("", ln).strip().strip("\"'`“”‘’")
        if ln:
            lines.append(ln)
    t = " ".join(lines)
    t = re.sub(r"\s+", " ", t).strip().rstrip(".,;:!").strip()
    t = t.upper()
    if t in {"NONE", "NO TEXT", "N/A", "UNREADABLE", "EMPTY"}:
        return ""
    # Safety net: if the model leaked a state banner/slogan next to a plate number, drop it.
    if re.search(r"\d", t):
        stripped = re.sub(r"\s+", " ", _BANNER_RE.sub(" ", t)).strip()
        if stripped and re.search(r"\d", stripped):
            t = stripped
    return t


def normalize(s):
    """Official grading normalization: uppercase, drop whitespace and - . · _"""
    s = (s or "").upper()
    s = re.sub(r"\s+", "", s)
    return re.sub(r"[-.·_]", "", s)


# ----------------------------------------------------------------------------- model
MARKER = ".download_complete"


def resolve_model_path():
    """Use local weights if fully downloaded, otherwise download them (resumes partial downloads)."""
    if os.path.isfile(os.path.join(MODEL_DIR, MARKER)):
        return MODEL_DIR
    from huggingface_hub import snapshot_download
    kw = dict(ignore_patterns=["*.bin", "*.pt", "*.pth", "*.gguf", "*.onnx", "original/*"], max_workers=16)
    try:
        path = snapshot_download(MODEL_ID, local_dir=MODEL_DIR, **kw)
    except OSError:                                                # read-only dir -> HF cache
        return snapshot_download(MODEL_ID, **kw)
    with open(os.path.join(path, MARKER), "w") as f:
        f.write(MODEL_ID)
    return path


class OCREngine:
    def __init__(self):
        import torch
        from transformers import AutoModelForImageTextToText, AutoProcessor
        self.torch = torch
        path = resolve_model_path()
        self.processor = AutoProcessor.from_pretrained(path)
        self.model = AutoModelForImageTextToText.from_pretrained(
            path, dtype=torch.bfloat16, device_map="cuda", attn_implementation="sdpa",
        ).eval()

    def _read(self, im):
        torch = self.torch
        messages = [{"role": "user", "content": [
            {"type": "image", "image": im},
            {"type": "text", "text": PROMPT},
        ]}]
        inputs = self.processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt",
        ).to(self.model.device)
        with torch.inference_mode():
            out = self.model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                output_scores=True, return_dict_in_generate=True,
            )
        seq = out.sequences[0, inputs["input_ids"].shape[1]:]
        raw = self.processor.decode(seq, skip_special_tokens=True)
        logps = []
        for step, tok in zip(out.scores, seq):
            p = torch.softmax(step[0].float(), dim=-1)[tok].item()
            logps.append(math.log(max(p, 1e-9)))
        conf = math.exp(sum(logps) / len(logps)) if logps else 0.0
        return postprocess(raw), conf, raw

    def predict(self, path):
        im = load_image(path)
        text, conf, raw = self._read(im)
        if not text or conf < 0.80 or needs_enhance(im):
            t2, c2, r2 = self._read(enhance(im))
            if t2 and (not text or c2 > conf):
                text, conf, raw = t2, c2, r2
        return {"text": text, "confidence": round(float(conf), 4), "raw": raw}

    def warmup(self):
        im = Image.new("RGB", (448, 224), "white")
        self._read(im)

In [ ]:
%%writefile server.py
"""Started by the container CMD: loads the model once, then answers app.py over localhost TCP."""
import json, os, socketserver, sys, threading, time, traceback

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
HOST = "127.0.0.1"
PORT = int(os.environ.get("OCR_PORT", "8765"))

READY = threading.Event()
LOCK = threading.Lock()
ENGINE = None


class Handler(socketserver.StreamRequestHandler):
    def handle(self):
        try:
            req = json.loads(self.rfile.readline().decode() or "{}")
            if req.get("cmd") == "ping":
                resp = {"ok": True, "ready": READY.is_set()}
            else:
                if not READY.wait(timeout=float(req.get("wait", 25))):
                    resp = {"ok": False, "error": "model not ready"}
                else:
                    with LOCK:
                        resp = {"ok": True, **ENGINE.predict(req["image"])}
        except Exception as e:
            traceback.print_exc()
            resp = {"ok": False, "error": repr(e)}
        self.wfile.write((json.dumps(resp, ensure_ascii=False) + "\n").encode())


class Server(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


def main():
    global ENGINE
    try:
        srv = Server((HOST, PORT), Handler)
    except OSError:
        print("[server] port busy - another server is already running", flush=True)
        return
    threading.Thread(target=srv.serve_forever, daemon=True).start()   # accept pings while loading
    t0 = time.time()
    print("[server] loading model ...", flush=True)
    from ocr_core import OCREngine
    for attempt in range(1, 6):                    # retry on flaky network during the weight download
        try:
            ENGINE = OCREngine()
            break
        except Exception:
            traceback.print_exc()
            print(f"[server] load attempt {attempt} failed, retrying ...", flush=True)
            time.sleep(5)
    else:
        sys.exit(1)
    ENGINE.warmup()
    READY.set()
    print(f"[server] ready in {time.time() - t0:.1f}s on {HOST}:{PORT}", flush=True)
    while True:
        time.sleep(3600)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile app.py
"""Harness entry point:  python3 /app/app.py --input-image /app/input/image_01.png
Writes /app/output/<name>_output.json  ->  {"text": "...", "confidence": 0.94}"""
import argparse, json, os, socket, subprocess, sys, time
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
HOST, PORT = "127.0.0.1", int(os.environ.get("OCR_PORT", "8765"))
T0 = time.time()
BUDGET = 26.0  # seconds; harness limit is 30 s per image - never exceed it (that zeroes the whole run)


def ask(payload, timeout):
    with socket.create_connection((HOST, PORT), timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall((json.dumps(payload) + "\n").encode())
        buf = b""
        while not buf.endswith(b"\n"):
            chunk = s.recv(65536)
            if not chunk:
                break
            buf += chunk
    return json.loads(buf.decode())


def start_server_background():
    subprocess.Popen(
        [sys.executable, os.path.join(HERE, "server.py")],
        stdout=open("/tmp/ocr_server.log", "a"), stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True,
    )


def run(image_path):
    """Ask the model server (started by the container CMD). If it isn't running, start it and keep
    waiting. If the model is still downloading/loading when the budget runs out, raise -> main()
    writes an empty answer: losing one image is far better than blowing the 30 s limit."""
    spawned = False
    while True:
        left = BUDGET - (time.time() - T0)
        if left < 1.0:
            raise TimeoutError("model not ready within the per-image budget")
        try:
            r = ask({"image": image_path, "wait": max(0.5, left - 1.0)}, timeout=left)
        except (ConnectionRefusedError, FileNotFoundError):
            if not spawned:
                start_server_background()
                spawned = True
            time.sleep(1.0)
            continue
        if r.get("ok"):
            return r
        raise RuntimeError(r.get("error", "server error"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input-image", required=True)
    ap.add_argument("--output-dir", default=os.environ.get("OCR_OUTPUT_DIR", "/app/output"))
    args = ap.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    out_path = os.path.join(args.output_dir, Path(args.input_image).stem + "_output.json")
    result = {"text": "", "confidence": 0.0}
    try:
        r = run(os.path.abspath(args.input_image))
        result = {"text": str(r.get("text", "")), "confidence": float(r.get("confidence", 0.0))}
    except Exception as e:
        print(f"[app] error: {e!r}", file=sys.stderr)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False)
    print(json.dumps(result, ensure_ascii=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile download_model.py
"""Download the weights to OCR_MODEL_DIR (the server does the same automatically at container start)."""
from ocr_core import MODEL_ID, resolve_model_path

print("downloaded", MODEL_ID, "->", resolve_model_path())

In [ ]:
%%writefile Dockerfile
# Final stage MUST start from the mandated ROCm base. Never use --squash / FROM scratch.
FROM rocm/pytorch:rocm10.0_ubuntu26.04_py3.14_pytorch_release_2.13.0

ARG OCR_MODEL_ID=Qwen/Qwen3-VL-8B-Instruct
ENV PYTHONUNBUFFERED=1 \
    PIP_BREAK_SYSTEM_PACKAGES=1 \
    HF_HUB_DISABLE_TELEMETRY=1 \
    HF_XET_HIGH_PERFORMANCE=1 \
    TOKENIZERS_PARALLELISM=false \
    OCR_MODEL_ID=${OCR_MODEL_ID} \
    OCR_MODEL_DIR=/models/ocr-vlm \
    OCR_OUTPUT_DIR=/app/output

WORKDIR /app
COPY requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

# Weights are NOT baked in (keeps the push small). server.py downloads them from the
# Hugging Face Hub into /models at container start, inside the 10-minute startup budget.
# To bake them instead, uncomment:  RUN python3 /app/download_model.py
COPY app.py ocr_core.py server.py download_model.py /app/
RUN mkdir -p /app/input /app/output /models && chmod 777 /models

# Download (first start) + load the model once; app.py talks to it on 127.0.0.1:8765
CMD ["python3", "/app/server.py"]

In [ ]:
%%writefile .dockerignore
samples/
output/
input/
__pycache__/
*.ipynb
*.zip
.env
*.log

## 3. Install dependencies & download weights

`pip install` inside a session only lasts for that session — re-run this cell each new day.
Weights go to persistent storage, so they download only once (~17 GB of your 25 GB).

In [ ]:
!pip install -q -r requirements.txt
try:
    import torchvision
    print("torchvision", torchvision.__version__, "OK")
except ImportError:
    print("WARNING: torchvision missing. Qwen-VL processors need it. Install the ROCm build matching torch,"
          " e.g.  pip install torchvision --index-url https://download.pytorch.org/whl/<rocm-version> --no-deps")
import transformers; print("transformers", transformers.__version__)

In [ ]:
t0 = time.time()
!python3 download_model.py
print(f"done in {time.time()-t0:.0f}s");
!du -sh {MODEL_DIR}

## 4. Recreate the sample images from the PDF

The 10 sample images are embedded below and saved in the same formats as the PDF (PNG / JPEG / TIFF).
They're illustrative only — the graded set is different and harder, so never hard-code answers.

In [ ]:
import base64, io
from PIL import Image

EXPECTED = {
    "image_01.png": "7ABC123",
    "image_02.png": "京 A·12345",
    "image_03.jpg": "JHT 2951",
    "image_04.png": "5XYZ891",
    "image_05.jpg": "沪 B·88888",
    "image_06.png": "STOP",
    "image_07.tif": "STOP",
    "image_08.jpg": "SPEED LIMIT 65",
    "image_09.png": "ROAD WORK AHEAD",
    "image_10.tif": "35"
}
_B64 = {
    "image_01.png": "...",
    "image_02.png": "...",
    "image_03.jpg": "...",
    "image_04.png": "...",
    "image_05.jpg": "...",
    "image_06.png": "...",
    "image_07.tif": "...",
    "image_08.jpg": "...",
    "image_09.png": "...",
    "image_10.tif": "..."
}
for name, b in _B64.items():
    im = Image.open(io.BytesIO(base64.b64decode(b))).convert("RGB")
    fmt = {"png": "PNG", "jpg": "JPEG", "tif": "TIFF"}[name.rsplit(".", 1)[1]]
    im.save(SAMPLES / name, fmt, **({"quality": 95} if fmt == "JPEG" else {}))
json.dump(EXPECTED, open(SAMPLES / "expected.json", "w"), ensure_ascii=False, indent=1)
print(sorted(p.name for p in SAMPLES.iterdir()))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, name in zip(axes.flat, EXPECTED):
    ax.imshow(Image.open(SAMPLES / name).convert("RGB")); ax.set_title(f"{name}\n{EXPECTED[name]}"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Evaluate on the samples (official normalization)

Pass/fail per image after: uppercase → remove whitespace → remove `- . · _`.

In [ ]:
import importlib, ocr_core
importlib.reload(ocr_core)
from ocr_core import OCREngine, normalize

t0 = time.time()
engine = OCREngine()
engine.warmup()
print(f"model loaded in {time.time()-t0:.1f}s | VRAM allocated {torch.cuda.memory_allocated()/2**30:.1f} GiB")

In [ ]:
def evaluate(engine, items, verbose=True):
    passed, rows = 0, []
    for name, path, expected in items:
        t0 = time.time()
        r = engine.predict(str(path))
        dt = time.time() - t0
        ok = normalize(r["text"]) == normalize(expected)
        passed += ok
        rows.append((name, ok, r["text"], expected, r["confidence"], dt))
        if verbose:
            print(f"{'PASS' if ok else 'FAIL'}  {name:28s} got={r['text']!r:24s} expected={expected!r:20s} "
                  f"conf={r['confidence']:.2f}  {dt:.2f}s" + ("" if ok else f"   raw={r['raw']!r}"))
    print(f"\n==> {passed}/{len(items)} correct  ({passed*20} / {len(items)*20} pts)")
    return rows

importlib.reload(ocr_core)          # pick up prompt / post-processing edits without reloading weights
engine.__class__ = ocr_core.OCREngine
rows = evaluate(engine, [(n, SAMPLES / n, e) for n, e in EXPECTED.items()])

## 6. Robustness test — build for generality

The hidden set is harder (noise, blur, glare, low light, off-axis, odd sizes). This applies synthetic
degradations to every sample and re-scores. Use it to tune `PROMPT`, `MAX_SIDE`, `enhance()` in `ocr_core.py`
or to compare models (change `MODEL_ID` in section 1, re-download, re-run).

In [ ]:
import random
import numpy as np
from PIL import ImageFilter, ImageEnhance, ImageDraw

AUG = PROJ / "aug"; AUG.mkdir(exist_ok=True)
rng = np.random.default_rng(0)

def a_blur(im):    return im.filter(ImageFilter.GaussianBlur(2.5))
def a_motion(im, k=9):
    a = np.asarray(im).astype(np.float32)
    out = np.zeros_like(a)
    for s in range(-(k // 2), k // 2 + 1):
        out += np.roll(a, s, axis=1)          # horizontal streak, like a passing car
    return Image.fromarray((out / k).clip(0, 255).astype(np.uint8))
def a_noise(im):
    a = np.asarray(im).astype(np.float32) + rng.normal(0, 40, (im.height, im.width, 3))
    return Image.fromarray(a.clip(0, 255).astype(np.uint8))
def a_dark(im):    return ImageEnhance.Brightness(im).enhance(0.25)
def a_glare(im):
    g = Image.new("L", im.size, 0); d = ImageDraw.Draw(g)
    w, h = im.size; d.ellipse([w*0.45, -h*0.3, w*1.1, h*0.7], fill=200)
    g = g.filter(ImageFilter.GaussianBlur(min(w, h)//6))
    return Image.composite(Image.new("RGB", im.size, "white"), im, g)
def a_skew(im):
    w, h = im.size
    return im.transform(im.size, Image.PERSPECTIVE, (1, 0.25, -0.12*w, 0.05, 1, -0.02*h, 0.0004, 0), Image.BICUBIC, fillcolor=(90, 90, 90))
def a_rotate(im):  return im.rotate(12, expand=True, fillcolor=(90, 90, 90))
def a_huge(im):    return im.resize((im.width*6, im.height*6), Image.BICUBIC)
def a_tiny(im):    return im.resize((max(40, im.width//5), max(20, im.height//5)), Image.BILINEAR)
def a_combo(im):   return a_noise(a_dark(a_blur(a_skew(im))))

AUGS = dict(blur=a_blur, motion=a_motion, noise=a_noise, dark=a_dark, glare=a_glare,
            skew=a_skew, rotate=a_rotate, huge=a_huge, tiny=a_tiny, combo=a_combo)
items = []
for n, e in EXPECTED.items():
    base = Image.open(SAMPLES / n).convert("RGB")
    for an, fn in AUGS.items():
        ext = random.Random(an + n).choice(["png", "jpg", "tif"])
        p = AUG / f"{pathlib.Path(n).stem}_{an}.{ext}"
        fn(base).save(p)
        items.append((p.name, p, e))
aug_rows = evaluate(engine, items, verbose=False)
for an in AUGS:
    r = [x for x in aug_rows if f"_{an}." in x[0]]
    print(f"{an:8s} {sum(x[1] for x in r)}/{len(r)}   max {max(x[5] for x in r):.2f}s")
print("\nFailures:")
for name, ok, got, exp, conf, dt in aug_rows:
    if not ok: print(f"  {name:28s} got={got!r:24s} expected={exp!r}")

## 7. End-to-end test of the exact harness contract

Frees the in-notebook model, starts `server.py` the way the container `CMD` will, then calls
`python3 app.py --input-image …` once per image as a **separate process** — just like the grader —
measuring per-image time and peak VRAM (must stay between 1 and 48 GiB; per image < 30 s; all 10 < 10 min).

In [ ]:
import gc, threading, socket
del engine; gc.collect(); torch.cuda.empty_cache()

def vram_used_gib():
    try:
        out = subprocess.run(["amd-smi", "metric", "--mem", "--json"], capture_output=True, text=True, timeout=10).stdout
        j = json.loads(out); j = j[0] if isinstance(j, list) else j
        if isinstance(j, dict) and "gpu_data" in j: j = j["gpu_data"][0]
        u = j["mem_usage"]["used_vram"]
        v, unit = (u["value"], u.get("unit", "MB")) if isinstance(u, dict) else (u, "MB")
        return float(v) / (1024 if unit.upper().startswith("M") else 1)
    except Exception:
        try:
            out = subprocess.run(["rocm-smi", "--showmeminfo", "vram", "--json"], capture_output=True, text=True, timeout=10).stdout
            card = next(iter(json.loads(out).values()))
            return int(card["VRAM Total Used Memory (B)"]) / 2**30
        except Exception:
            return float("nan")

peak = {"v": 0.0}; stop = threading.Event()
def monitor():
    while not stop.is_set():
        v = vram_used_gib()
        if v == v: peak["v"] = max(peak["v"], v)
        time.sleep(3)
threading.Thread(target=monitor, daemon=True).start()

t0 = time.time()
server = subprocess.Popen([sys.executable, "server.py"], stdout=open(PROJ / "server.log", "w"), stderr=subprocess.STDOUT)
while True:
    try:
        with socket.create_connection(("127.0.0.1", 8765), timeout=2) as s:
            s.sendall(b'{"cmd":"ping"}\n'); r = json.loads(s.makefile().readline())
        if r.get("ready"): break
    except OSError:
        pass
    if server.poll() is not None: print(open(PROJ / "server.log").read()); raise RuntimeError("server died")
    time.sleep(2)
print(f"startup (model load + warmup): {time.time()-t0:.1f}s   (limit 600s)")

In [ ]:
shutil.rmtree(OUTDIR, ignore_errors=True); OUTDIR.mkdir()
total0, ok, worst = time.time(), 0, 0.0
for name, exp in EXPECTED.items():
    t0 = time.time()
    subprocess.run([sys.executable, "app.py", "--input-image", str(SAMPLES / name)], check=True, capture_output=True)
    dt = time.time() - t0; worst = max(worst, dt)
    out_file = OUTDIR / (pathlib.Path(name).stem + "_output.json")
    res = json.load(open(out_file, encoding="utf-8"))
    assert set(res) <= {"text", "confidence"} and isinstance(res["text"], str)
    good = normalize(res["text"]) == normalize(exp); ok += good
    print(f"{'PASS' if good else 'FAIL'} {out_file.name:28s} {json.dumps(res, ensure_ascii=False):45s} {dt:5.2f}s")
total = time.time() - total0
print(f"\n{ok}/10 correct | worst image {worst:.1f}s (limit 30s) | total {total:.1f}s (limit 600s) | peak VRAM {peak['v']:.1f} GiB (1-48 GiB)")

**Fallback check:** what happens if the server is *not* running (e.g. harness overrides the CMD)?
`app.py` starts the server itself and waits for it (weights already on disk -> it should load in time);
it never runs past 30 s - worst case it writes an empty answer for that one image.

In [ ]:
server.terminate(); server.wait(); time.sleep(2)
t0 = time.time()
r = subprocess.run([sys.executable, "app.py", "--input-image", str(SAMPLES / "image_07.tif")], capture_output=True, text=True)
print(r.stdout.strip(), f"\nno-server case took {time.time()-t0:.1f}s (must be < 30s)")
time.sleep(1)
t0 = time.time()   # the background server app.py just spawned should now serve the next one
r = subprocess.run([sys.executable, "app.py", "--input-image", str(SAMPLES / "image_10.tif")], capture_output=True, text=True)
print(r.stdout.strip(), f"\nnext image took {time.time()-t0:.1f}s")

In [ ]:
# clean up: stop background server + monitor, free the GPU
stop.set()
!pkill -f "server.py" || true
print("stopped")

**Cold-start check (weights downloaded at runtime):** simulates a fresh container — no server, no weights
on disk. Checks that `app.py` returns within 30 s while the model downloads, and how long download + load take
(startup budget 600 s). To avoid needing space for a second copy, it **deletes the model folder and re-downloads
it into the same place** (so it's restored afterwards). Your pod's network may be slower than the grader's datacenter.
Set `RUN_COLD_TEST = True` to run it.

In [ ]:
RUN_COLD_TEST = False

if RUN_COLD_TEST:
    !pkill -f "server.py" || true
    time.sleep(2)
    shutil.rmtree(MODEL_DIR, ignore_errors=True)
    print("free disk:", shutil.disk_usage(PERSIST).free // 2**30, "GiB (need ~18 GiB)")
    open("/tmp/ocr_server.log", "w").close()

    t0 = time.time()
    r = subprocess.run([sys.executable, "app.py", "--input-image", str(SAMPLES / "image_01.png")], capture_output=True, text=True)
    print(r.stdout.strip(), f"  <- image sent while downloading took {time.time()-t0:.1f}s (must be < 30s)")

    def dir_gib(d):
        return sum(f.stat().st_size for f in pathlib.Path(d).rglob("*") if f.is_file()) / 2**30 if pathlib.Path(d).exists() else 0.0

    ready, last = False, 0
    while time.time() - t0 < 900:
        try:
            with socket.create_connection(("127.0.0.1", 8765), timeout=2) as s:
                s.sendall(b'{"cmd":"ping"}\n'); ready = bool(json.loads(s.makefile().readline()).get("ready"))
        except OSError:
            ready = False
        if ready: break
        if time.time() - last > 30:
            last = time.time(); print(f"  {time.time()-t0:5.0f}s  downloaded {dir_gib(MODEL_DIR):5.1f} GiB")
        time.sleep(5)

    if ready:
        el = time.time() - t0
        print(f"READY after {el:.0f}s -> {'OK' if el < 600 else 'TOO SLOW here (grader datacenter should be faster)'}")
        r = subprocess.run([sys.executable, "app.py", "--input-image", str(SAMPLES / "image_01.png")], capture_output=True, text=True)
        print("next image:", r.stdout.strip())
    else:
        print("NOT READY after 15 min. Server log:")
        print(open("/tmp/ocr_server.log").read()[-3000:])
    !pkill -f "server.py" || true

## 8. Package the project

Downloads a zip with everything needed to build the image (no samples, no weights, no secrets).
Right-click `ocr_submission.zip` in the JupyterLab file browser → **Download**.

In [ ]:
zip_path = PERSIST / "ocr_submission.zip"
files = ["app.py", "ocr_core.py", "server.py", "download_model.py", "requirements.txt", "Dockerfile", ".dockerignore"]
import zipfile
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(PROJ / f, f"ocr_submission/{f}")
print(zip_path, f"{zip_path.stat().st_size/1024:.1f} KB")
import re
hits = [(f, m.group(0)) for f in files for m in re.finditer(r"hf_[A-Za-z0-9]{20,}|api[_-]?key\s*=|token\s*=\s*['\"]", (PROJ / f).read_text())]
print("secrets found:", hits) if hits else print("no secrets found")

## 9. Submission guide

The notebook session has no Docker, so build on any machine with Docker (your laptop, or an AMD Developer
Cloud VM). Building needs **no GPU**. The weights are not in the image, so you only push your own small layers
(pip packages + code, typically < 1 GB) — the ROCm base layers already exist on Docker Hub and are not re-uploaded.

### 9.1 Build
```bash
unzip ocr_submission.zip && cd ocr_submission
docker build -t ocr-amd:v1 .
# never add --squash; never flatten/export-import the image
```

### 9.2 Verify the hard gates (all three = automatic zero if they fail)
```bash
# Gate 1 - built on the mandated base (base layers must be the bottom layers of yours)
docker pull rocm/pytorch:rocm10.0_ubuntu26.04_py3.14_pytorch_release_2.13.0
docker inspect -f '{{json .RootFS.Layers}}' rocm/pytorch:rocm10.0_ubuntu26.04_py3.14_pytorch_release_2.13.0 > base.txt
docker inspect -f '{{json .RootFS.Layers}}' ocr-amd:v1 > mine.txt
python3 -c "import json;b=json.load(open('base.txt'));m=json.load(open('mine.txt'));print('BASE OK' if m[:len(b)]==b else 'BASE MISMATCH')"

# Gate 2 - uncompressed size < 60 GiB
docker image inspect ocr-amd:v1 --format '{{.Size}}' | awk '{printf "%.1f GiB\n", $1/2^30}'
docker history --no-trunc --format '{{.Size}}' ocr-amd:v1
```

### 9.3 Gate 3 - run it like the grader (on a machine with an AMD GPU)
```bash
docker run -d --name ocrtest --device=/dev/kfd --device=/dev/dri --group-add video \
  --security-opt seccomp=unconfined --shm-size 8g ocr-amd:v1
docker logs -f ocrtest            # first start downloads ~17 GB, then "[server] ready in XXs" - must be < 10 min

docker cp samples/. ocrtest:/app/input/      # or any png/jpg/tif you have
for f in image_01.png image_03.jpg image_07.tif; do
  /usr/bin/time -f "%es" docker exec ocrtest python3 /app/app.py --input-image /app/input/$f
done
docker exec ocrtest sh -c 'cat /app/output/*_output.json'
watch -n1 'amd-smi metric --mem | grep USED_VRAM'     # in another terminal: 1-48 GiB
docker rm -f ocrtest
```
If you don't have AMD hardware locally, section 7 of this notebook already exercises the identical code path on the AMD GPU.

### 9.4 Push to a public registry
Docker Hub (free) example:
```bash
docker login                                   # credentials stay on your machine, never in the image
docker tag ocr-amd:v1 <dockerhub-user>/<any-name>:v1
docker push <dockerhub-user>/<any-name>:v1
```
Then make the repository **Public** (Docker Hub → repo → Settings → Visibility).
GHCR alternative: `docker tag ocr-amd:v1 ghcr.io/<user>/<name>:v1 && docker push …`, then set the package visibility to Public.

Confirm it's pullable **without credentials**:
```bash
docker logout && docker pull <dockerhub-user>/<any-name>:v1
```

### 9.5 Submit
Submit the image reference (e.g. `docker.io/<user>/<name>:v1`, or better the immutable digest
`<user>/<name>@sha256:…` from `docker inspect --format '{{index .RepoDigests 0}}' <user>/<name>:v1`)
on the lablab.ai challenge page: https://lablab.ai/ai-hackathons/amd-lablab-ai-academy-challenge

### Final checklist
- [ ] `FROM rocm/pytorch:rocm10.0_ubuntu26.04_py3.14_pytorch_release_2.13.0`, no squash
- [ ] Image < 60 GiB uncompressed
- [ ] `/app/app.py` accepts `--input-image`, writes `/app/output/<stem>_output.json` with `{"text", "confidence"}`
- [ ] Works for PNG, JPEG and TIFF; huge and tiny images
- [ ] Startup (incl. weight download) < 10 min, each image < 30 s, all 10 < 10 min
- [ ] VRAM between 1 and 48 GiB while running
- [ ] No `.env`, API keys or tokens in the image; image reference **not** committed to any public repo
- [ ] Image is public and pulls without login